**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Scaling Neural Networks

The [ANN](../Intro_ANN/Intro_ANN.ipynb) and [CNN](../Intro_CNN/Intro_CNN.ipynb) workshops trained models with thousands of parameters. Frontier models have *billions* — and the jump is not "the same but bigger": it changes what limits you (memory and data movement, not ideas), what you measure (throughput, utilization), and even what to expect (scaling laws). This workshop builds that systems mindset with experiments you can run on a laptop CPU.

> ℹ️ All benchmarks below run on **CPU** and demonstrate the *reasoning*; sections that only make sense on GPUs are clearly marked *illustrative — not executed here*.

## 0. Introduction

Three questions organize everything:

1. **Where do the parameters and FLOPs go?** (accounting)
2. **What limits my throughput?** (compute vs memory bandwidth — the [GPU workshop's](../../Intro_GPU/Intro_GPU.ipynb) CGMA ratio, at training scale)
3. **What does more compute buy?** (scaling laws)

## 1. Pre-requisites

- [Intro to PyTorch](../../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb) and [Intro to ANN](../Intro_ANN/Intro_ANN.ipynb).
- [Intro to GPU Systems](../../Intro_GPU/README.md) — the memory-latency worldview.
- `pip install torch` (CPU build is fine here).

In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
print(torch.__version__, "| threads:", torch.get_num_threads())

2.13.0+cpu | threads: 24


---
### 🕐 Session 1 of 2 — *Why Scale? Accounting & Scaling Laws* (~35 min)
**Goal:** count parameters and FLOPs; see diminishing-but-predictable returns in a toy scaling study.
**Builds on:** [ANN](../Intro_ANN/Intro_ANN.ipynb). &nbsp; **Feeds into:** Session 2 (making training fast).

---

## 2. Parameter & FLOP Accounting

💡 **Intuition.** Before optimizing anything, learn to *count*. A linear layer $d_{in} \to d_{out}$ stores $d_{in} d_{out} + d_{out}$ weights and spends $\approx 2 \, d_{in} d_{out}$ FLOPs per input (one multiply + one add per weight). Rule of thumb for training: **forward ≈ 2 FLOPs/param/token, backward ≈ twice the forward** — so training cost $\approx 6 \times$ params $\times$ tokens. That one line explains most headlines about GPU-months.

In [2]:
def account(model, x):
    n_params = sum(p.numel() for p in model.parameters())
    flops_fwd = 0
    for m in model.modules():
        if isinstance(m, nn.Linear):
            flops_fwd += 2 * m.in_features * m.out_features
    print(f"params: {n_params:>10,}   fwd FLOPs/sample: {flops_fwd:>12,}   train ≈ {3*flops_fwd:,} FLOPs/sample")
    return n_params

for width in [64, 256, 1024]:
    print(f"width {width:>5}: ", end="")
    account(nn.Sequential(nn.Linear(128, width), nn.ReLU(),
                          nn.Linear(width, width), nn.ReLU(),
                          nn.Linear(width, 1)), None)
print("→ doubling width ≈ 4x params and FLOPs: cost grows QUADRATICALLY in width")

width    64: params:     12,481   fwd FLOPs/sample:       24,704   train ≈ 74,112 FLOPs/sample
width   256: params:     99,073   fwd FLOPs/sample:      197,120   train ≈ 591,360 FLOPs/sample
width  1024: params:  1,182,721   fwd FLOPs/sample:    2,361,344   train ≈ 7,084,032 FLOPs/sample
→ doubling width ≈ 4x params and FLOPs: cost grows QUADRATICALLY in width


## 3. A Toy Scaling Study

💡 **Intuition.** The famous scaling-law plots (loss vs compute, straight lines on log-log axes) are *empirical* — but you can reproduce their shape on a laptop. Fix a task, sweep model size with an equal training budget per size, and plot final loss vs parameters on log axes. Expect: big early gains, then a steady power-law-ish slide — and eventually a floor set by the data's intrinsic noise, which **no** amount of scale removes.

In [3]:
# Task: regress y = sin(4x) + noise. The noise floor (var 0.01) is unbeatable BY DESIGN.
rng = np.random.default_rng(0)
Xd = rng.uniform(-1, 1, (4096, 1)).astype(np.float32)
yd = np.sin(4 * Xd) + 0.1 * rng.standard_normal((4096, 1)).astype(np.float32)
Xt, yt = torch.from_numpy(Xd), torch.from_numpy(yd)

def train_width(width, steps=600):
    model = nn.Sequential(nn.Linear(1, width), nn.Tanh(),
                          nn.Linear(width, width), nn.Tanh(),
                          nn.Linear(width, 1))
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(steps):
        idx = torch.randint(0, len(Xt), (256,))
        loss = ((model(Xt[idx]) - yt[idx])**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return sum(p.numel() for p in model.parameters()), ((model(Xt) - yt)**2).mean().item()

widths = [1, 2, 4, 8, 16, 64, 256]
results = [train_width(w) for w in widths]
for w, (p, l) in zip(widths, results): print(f"width {w:>4}  params {p:>6,}  final MSE {l:.4f}")

width    1  params      6  final MSE 0.2720
width    2  params     13  final MSE 0.0219
width    4  params     33  final MSE 0.0108
width    8  params     97  final MSE 0.0105
width   16  params    321  final MSE 0.0105
width   64  params  4,353  final MSE 0.0103
width  256  params 66,561  final MSE 0.0112


In [4]:
ps, ls = zip(*results)
plt.figure(figsize=(7, 3))
plt.loglog(ps, ls, "o-", label="final training MSE")
plt.axhline(0.01, color="k", linestyle="--", linewidth=0.9, label="noise floor (σ²=0.01)")
plt.xlabel("parameters"); plt.ylabel("MSE"); plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.title("A laptop-scale scaling curve: power-law-ish descent onto the noise floor")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1901664/698132217.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Real scaling laws sweep **data and compute** jointly (Chinchilla's lesson: params and tokens should grow together) — but the qualitative structure you just plotted is the same one governing billion-dollar training runs.

---
### 🕐 Session 2 of 2 — *Making Training Fast* (~40 min)
**Goal:** find the actual bottleneck: batch-size throughput curves, profiling, gradient accumulation, precision.
**Builds on:** Session 1.

---

## 4. Throughput vs Batch Size

💡 **Intuition.** Per-sample overhead (Python, kernel launches, optimizer bookkeeping) is *fixed*; compute grows with the batch. Small batches ⇒ overhead dominates and throughput climbs as batches grow; eventually arithmetic saturates the hardware and the curve flattens — or even *dips*, as very large batches spill out of cache (CPU) or run out of memory (GPU). **Measure the knee** — that's your efficient operating point.

In [5]:
model = nn.Sequential(nn.Linear(256, 512), nn.ReLU(),
                      nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

def throughput(bs, iters=30):
    x = torch.randn(bs, 256); y = torch.randint(0, 10, (bs,))
    for _ in range(3):                       # warmup (first-touch costs — remember the OS workshop!)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    tic = time.perf_counter()
    for _ in range(iters):
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    return bs * iters / (time.perf_counter() - tic)

sizes = [1, 4, 16, 64, 256, 1024]
tps = [throughput(b) for b in sizes]
for b, t in zip(sizes, tps): print(f"batch {b:>5}: {t:>10,.0f} samples/s")

batch     1:      2,678 samples/s
batch     4:     15,707 samples/s
batch    16:     52,695 samples/s
batch    64:    140,170 samples/s
batch   256:    274,296 samples/s
batch  1024:    231,011 samples/s


In [6]:
plt.figure(figsize=(7, 2.8))
plt.semilogx(sizes, tps, "o-")
plt.xlabel("batch size"); plt.ylabel("samples/s"); plt.grid(True, which="both", alpha=0.3)
plt.title("Throughput vs batch size: find the knee (numbers vary by machine)")
plt.tight_layout(); plt.show()

/tmp/ipykernel_1901664/1497128088.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Profile Before You Optimize

Guessing at bottlenecks is how you optimize the wrong thing. PyTorch ships a profiler — read its table before touching your code.

In [7]:
from torch.profiler import profile, ProfilerActivity

x = torch.randn(256, 256); y = torch.randint(0, 10, (256,))
with profile(activities=[ProfilerActivity.CPU]) as prof:
    for _ in range(10):
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=8))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
    autograd::engine::evaluate_function: AddmmBackward0         0.84%      96.957us        42.82%       4.972ms     165.720us            30  
                                         AddmmBackward0         0.63%      72.864us        40.29%       4.678ms     155.941us            30  
                                               aten::mm        38.70%       4.493ms        38.76%       4.500ms      89.990us            50  
                                           aten::linear         2.09%     243.018us        28.48%       3.307ms     110.236us            30  
      

USDT:2026-07-22 12:32:19 1901664:1901664 SyncActivityProfilerHandler.cpp:52] profiler_start
USDT:2026-07-22 12:32:19 1901664:1901664 SyncActivityProfilerHandler.cpp:59] profiler_stop


Reading it: `addmm`/`mm` rows are your matrix multiplies (real work); everything else is overhead. The ratio between them tells you whether to seek faster math or less overhead.

## 6. Gradient Accumulation

💡 **Intuition.** Want the optimization behavior of batch 1024 but only memory for 256? Run 4 micro-batches, **add up their gradients, step once**. Gradients are sums over samples, so the result is mathematically the large batch (identical when the loss averages per micro-batch of equal size and you scale by the count). This is the standard trick behind every 'effective batch size' line in a paper.

In [8]:
def grads_of(batches, accumulate):
    torch.manual_seed(7)
    m = nn.Linear(8, 1)
    if accumulate:
        m.zero_grad()
        for xb, yb in batches:
            (((m(xb) - yb)**2).mean() / len(batches)).backward()   # scale by micro-batch count
    else:
        xb = torch.cat([b[0] for b in batches]); yb = torch.cat([b[1] for b in batches])
        ((m(xb) - yb)**2).mean().backward()
    return m.weight.grad.clone()

data = [(torch.randn(4, 8), torch.randn(4, 1)) for _ in range(4)]
g_acc, g_big = grads_of(data, True), grads_of(data, False)
print("max |accumulated − full-batch gradient|:", (g_acc - g_big).abs().max().item())
assert torch.allclose(g_acc, g_big, atol=1e-6)
print("identical — accumulation IS the big batch, paid for in time instead of memory")

max |accumulated − full-batch gradient|: 2.384185791015625e-07
identical — accumulation IS the big batch, paid for in time instead of memory


## 7. The GPU-Scale Toolbox *(illustrative — not executed in this CPU notebook)*

On real accelerators the same reasoning continues with hardware-specific tools — presented here as a map, since none of this can be demonstrated honestly on CPU:

- **Mixed precision** (`torch.autocast` + fp16/bf16): tensor cores double-to-quadruple matmul throughput and halve activation memory; loss scaling guards small fp16 gradients.
- **Data parallelism** (`DistributedDataParallel`): replicate the model, split the batch, all-reduce gradients — gradient accumulation across machines, plus a network.
- **Memory arithmetic**: Adam training in fp32 costs ≈ 16 bytes/param (weights 4 + grads 4 + two moments 8) before activations — a 7B model wants ~112 GB, hence sharding (ZeRO/FSDP), activation checkpointing (recompute instead of store), and model/pipeline parallelism.

Each is the Session-2 mindset — *find the binding constraint, spend the cheap resource* — applied to a bigger machine.

## 8. Conclusion

Scaling is accounting plus bottleneck-hunting: count params and FLOPs, measure the throughput knee, profile before optimizing, accumulate when memory binds, and expect power-law returns onto a noise floor. The mindset transfers unchanged from laptop to cluster.

---
## Where next

- [Intro to GPU Systems](../../Intro_GPU/README.md) — the memory hierarchy all of this optimizes against.
- [Intro to Transformers](../../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb) — the architecture the scaling laws were measured on.
- [Intro to OS](../../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — first-touch, scheduling, and why your warmup iterations exist.